# BEIP — EDA 03: Census 2011 Demographics Analysis

**Objective:** Explore district-level demographics from `silver.census_demographics` and connect them to electoral patterns:
- District and state coverage in Census 2011
- Geographic distribution of literacy rates
- Demographic indicators: Sex ratio, SC/ST population concentrations, worker participation
- Cross-dataset correlation: Relationship between state-level literacy and voter turnout

## 1. Setup and Data Loading

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
from src.config import get_engine

engine = get_engine()
census = pd.read_sql("SELECT * FROM silver.census_demographics", engine)
print(f"Loaded {len(census):,} district records across {census['state_name'].nunique()} states/UTs.")

## 2. Basic Dataset Overview & Quality Checks

In [ ]:
census.head()

In [ ]:
print("=== Missing Values in Census Data ===")
print(census.isnull().sum())

print("\n=== Summary Statistics ===")
census.describe()

## 3. Literacy Rate Analysis
Examine literacy rate variation across states and district-level distribution.

In [ ]:
# State-level average literacy rate
state_lit = census.groupby("state_name")["literacy_rate"].mean().reset_index()
state_lit = state_lit.dropna().sort_values("literacy_rate", ascending=False)

plt.figure(figsize=(12, 10))
sns.barplot(data=state_lit, x="literacy_rate", y="state_name", palette="Blues_r")
plt.title("Average Literacy Rate (%) by State / UT (Census 2011)", fontsize=14, fontweight="bold")
plt.xlabel("Average Literacy Rate (%)")
plt.ylabel("State / UT")
plt.show()

print("Top 5 States by Literacy Rate:")
display(state_lit.head(5))
print("Bottom 5 States by Literacy Rate:")
display(state_lit.tail(5))

In [ ]:
# District Literacy Distribution
plt.figure(figsize=(10, 5))
sns.histplot(data=census, x="literacy_rate", bins=30, kde=True, color="royalblue")
plt.title("District Literacy Rate Distribution (Census 2011)", fontsize=13, fontweight="bold")
plt.xlabel("Literacy Rate (%)")
plt.ylabel("Number of Districts")
plt.show()

## 4. Demographic Metrics (Sex Ratio & SC/ST Distribution)
Analyze sex ratios and demographic composition across regions.

In [ ]:
state_demo = census.groupby("state_name").agg(
    avg_sex_ratio=("sex_ratio", "mean"),
    avg_sc_pct=("sc_percentage", "mean"),
    avg_st_pct=("st_percentage", "mean"),
    avg_worker_part=("worker_participation", "mean")
).reset_index()

plt.figure(figsize=(12, 10))
sns.barplot(data=state_demo.sort_values("avg_sex_ratio", ascending=False), x="avg_sex_ratio", y="state_name", palette="Purples_r")
plt.axvline(1000, color="red", linestyle="--", label="1:1 Ratio (1000 Females/1000 Males)")
plt.title("Average Sex Ratio (Females per 1,000 Males) by State", fontsize=14, fontweight="bold")
plt.xlabel("Sex Ratio (Females per 1000 Males)")
plt.ylabel("State / UT")
plt.legend()
plt.show()

## 5. Cross-Dataset Analysis: Literacy Rate vs. Voter Turnout
Join state-level Census demographics with state voter turnout from `silver.election_results` to observe structural patterns.

In [ ]:
# Load election turnout by state
elections = pd.read_sql("""
    SELECT 
        state_name, 
        AVG(turnout_percentage) AS avg_turnout
    FROM silver.election_results
    GROUP BY state_name
""", engine)

# Merge with state census
combined = state_lit.merge(elections, on="state_name", how="inner")

plt.figure(figsize=(12, 8))
sns.regplot(data=combined, x="literacy_rate", y="avg_turnout", scatter_kws={"s": 70, "alpha": 0.8}, line_kws={"color": "red"})

# Annotate points with state names
for _, row in combined.iterrows():
    plt.text(row["literacy_rate"] + 0.3, row["avg_turnout"] + 0.2, row["state_name"], fontsize=8, alpha=0.85)

corr_val = combined["literacy_rate"].corr(combined["avg_turnout"])
plt.title(f"State Literacy Rate vs. Average Voter Turnout (Pearson r = {corr_val:.2f})", fontsize=14, fontweight="bold")
plt.xlabel("State Literacy Rate (%)")
plt.ylabel("Average Voter Turnout (%)")
plt.show()